In [1]:
# Imports & Configuration
from pathlib import Path

import duckdb
import polars as pl
from great_tables import GT

# Change only this path
SOURCE = Path("../../raw_data/raw_orders_hu.csv").resolve().as_posix()

# DuckDB connection
con = duckdb.connect()

In [2]:
# ============================================================
# Functions
# ============================================================

def show_sql(query: str, max_rows: int = 100):
    """Execute DuckDB SQL and display results using Great Tables."""
    result = con.sql(query).limit(max_rows)

    df = pl.DataFrame(
        result.fetchall(),
        schema=result.columns,
        orient="row"
    )

    display(GT(df))

EXPECTED_CURRENCY = {
    "cz": "CZK",
    "sk": "EUR",
    "hu": "HUF"
}[Path(SOURCE).stem.split("_")[-1]]

In [3]:
# 01 | Source Overview

show_sql(f"""
    SUMMARIZE
    SELECT *
    FROM read_csv('{SOURCE}', all_varchar = true)
""")

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
order_id,VARCHAR,HU-300001,HU-301947,2006,None,None,None,None,None,3101,0.00
line_id,VARCHAR,14540000000000,14540000059390,2913,None,None,None,None,None,3101,0.00
created_at,VARCHAR,2025-12-31T23:02:18Z,2026-09-30T23:58:25Z,1907,None,None,None,None,None,3101,0.00
sku,VARCHAR,FB-BOARD-L,FB-WOK-30-BLK,31,None,None,None,None,None,3101,0.00
quantity,VARCHAR,1,2,2,None,None,None,None,None,3101,0.00
price_with_vat,VARCHAR,10590.0,8990.0,27,None,None,None,None,None,3101,0.00
discount_with_vat,VARCHAR,0.0,9845.0,125,None,None,None,None,None,3101,0.00
currency,VARCHAR,HUF,HUF,1,None,None,None,None,None,3101,0.00
customer_id,VARCHAR,hu-000001,hu-001252,1308,None,None,None,None,None,3101,5.42
cancelled_at,VARCHAR,2026-01-02T03:09:42Z,2026-10-02T20:42:40Z,63,None,None,None,None,None,3101,97.00


In [4]:
# 02 | Data Quality Checks

show_sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_rows,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(order_id), '') IS NULL
               OR NULLIF(TRIM(line_id), '') IS NULL
               OR NULLIF(TRIM(sku), '') IS NULL
               OR NULLIF(TRIM(created_at), '') IS NULL
               OR NULLIF(TRIM(exported_at), '') IS NULL
        ) AS missing_required,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(customer_id), '') IS NULL
        ) AS missing_customer,

        COUNT(*) FILTER (
            WHERE cancelled_at IS NOT NULL
        ) AS cancelled_rows,

        COUNT(*) FILTER (
            WHERE TRY_CAST(quantity AS INTEGER) IS NULL
               OR TRY_CAST(quantity AS INTEGER) <= 0
               OR TRY_CAST(price_with_vat AS DECIMAL(18,2)) IS NULL
               OR TRY_CAST(price_with_vat AS DECIMAL(18,2)) < 0
               OR TRY_CAST(discount_with_vat AS DECIMAL(18,2)) IS NULL
               OR TRY_CAST(discount_with_vat AS DECIMAL(18,2)) < 0
        ) AS invalid_numeric,

        COUNT(*) FILTER (
            WHERE currency IS DISTINCT FROM '{EXPECTED_CURRENCY}'
        ) AS invalid_currency,

        COUNT(*) FILTER (
            WHERE TRY_CAST(created_at AS TIMESTAMPTZ) IS NULL
               OR TRY_CAST(exported_at AS TIMESTAMP) IS NULL
               OR (
                   NULLIF(TRIM(cancelled_at), '') IS NOT NULL
                   AND TRY_CAST(cancelled_at AS TIMESTAMPTZ) IS NULL
               )
        ) AS invalid_timestamps

    FROM read_csv('{SOURCE}', all_varchar = true)
""")

total_rows,duplicate_rows,missing_required,missing_customer,cancelled_rows,invalid_numeric,invalid_currency,invalid_timestamps
3101,146,0,168,93,0,0,0


In [5]:
# 03 | Duplicate Investigation

show_sql(f"""
    WITH versions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY line_id
                ORDER BY exported_at DESC
            ) AS rn
        FROM read_csv(
            '{SOURCE}',
            all_varchar = true
        )
    )
    SELECT
        COUNT(*) AS duplicated_lines,
        COUNT(*) FILTER (
            WHERE old.cancelled_at IS DISTINCT FROM new.cancelled_at
        ) AS changed_cancellations,
        COUNT(*) FILTER (
            WHERE old.quantity IS DISTINCT FROM new.quantity
               OR old.price_with_vat IS DISTINCT FROM new.price_with_vat
               OR old.discount_with_vat IS DISTINCT FROM new.discount_with_vat
        ) AS changed_financial_values
    FROM versions old
    JOIN versions new
        ON old.line_id = new.line_id
       AND old.rn = 2
       AND new.rn = 1
""")

duplicated_lines,changed_cancellations,changed_financial_values
146,75,0


In [6]:
# 04 | Business Validity

show_sql(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE TRIM(order_id) = ''
               OR TRIM(line_id) = ''
               OR TRIM(sku) = ''
               OR TRIM(currency) = ''
        ) AS empty_required,

        COUNT(*) FILTER (
            WHERE TRY_CAST(discount_with_vat AS DECIMAL(18,2))
                > TRY_CAST(price_with_vat AS DECIMAL(18,2))
                * TRY_CAST(quantity AS INTEGER)
        ) AS excessive_discount,

        COUNT(*) FILTER (
            WHERE cancelled_at IS NOT NULL
              AND TRY_CAST(cancelled_at AS TIMESTAMPTZ)
                < TRY_CAST(created_at AS TIMESTAMPTZ)
        ) AS invalid_cancellation,

        COUNT(DISTINCT order_id) AS unique_orders,
        COUNT(DISTINCT sku) AS unique_skus

    FROM read_csv('{SOURCE}', all_varchar = true)
""")

empty_required,excessive_discount,invalid_cancellation,unique_orders,unique_skus
0,0,0,1947,32


In [7]:
# 05 | Schema

show_sql(f"""

    DESCRIBE
    SELECT *
    FROM read_csv_auto('{SOURCE}')
""")

column_name,column_type,null,key,default,extra
order_id,VARCHAR,YES,None,None,None
line_id,BIGINT,YES,None,None,None
created_at,TIMESTAMP WITH TIME ZONE,YES,None,None,None
sku,VARCHAR,YES,None,None,None
quantity,BIGINT,YES,None,None,None
price_with_vat,DOUBLE,YES,None,None,None
discount_with_vat,DOUBLE,YES,None,None,None
currency,VARCHAR,YES,None,None,None
customer_id,VARCHAR,YES,None,None,None
cancelled_at,TIMESTAMP WITH TIME ZONE,YES,None,None,None


In [8]:
# 06 | Sample Data

show_sql(f"""
    SELECT *
    FROM read_csv('{SOURCE}', all_varchar = true)
    LIMIT 5
""")

order_id,line_id,created_at,sku,quantity,price_with_vat,discount_with_vat,currency,customer_id,cancelled_at,exported_at
HU-300013,14540000000383,2025-12-31T23:15:46Z,FB-SPAT-SIL,1,2890.0,434.0,HUF,hu-000010,None,2026-01-01 02:00:58
HU-300013,14540000000403,2025-12-31T23:15:46Z,FB-PAN-20,1,19690.0,2954.0,HUF,hu-000010,None,2026-01-01 02:00:58
HU-300017,14540000000533,2025-12-31T23:28:18Z,FB-MITT,1,4390.0,658.0,HUF,hu-000005,None,2026-01-01 02:01:39
HU-300027,14540000000786,2025-12-31T23:52:08Z,FB-SAUCE-16,1,19690.0,2954.0,HUF,hu-000019,None,2026-01-01 02:02:29
HU-300027,14540000000815,2025-12-31T23:52:08Z,FB-PAN-24,1,22790.0,3418.0,HUF,hu-000019,None,2026-01-01 02:02:29


| Area | Finding | Recommendation |
|---|---|---|
| Duplicates | 146 duplicate rows, including 75 cancellation updates | Retain the latest export per `line_id` |
| Missing values | 168 missing customer IDs | Preserve NULL values |
| Cancellations | 93 cancelled rows in raw data | Handle in downstream calculations |
| Data validity | No invalid numeric values, timestamps or business values detected | Apply appropriate data types |
| Currency | HUF | Preserve source currency until conversion |

**Expected rows after deduplication:** 2,955.

**Pending checks:** SKU integrity against the product catalog and FX rate coverage.